## Implementation of Chat Memory w/ RAG-Fusion (RAM Persistence)

### Libraries, ChatOllama, Chroma vectorstore, LLM prompt initialization

In [1]:
from chromadb.config import Settings
from chromadb import Client
from langchain.vectorstores import Chroma
import chromadb

from langchain_ollama import ChatOllama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from typing_extensions import List, TypedDict
from langgraph.graph import START, StateGraph
from langgraph.checkpoint.memory import MemorySaver

import os, uuid
from datetime import datetime

date = datetime.today().strftime('%Y-%m-%d')
proj_name = "Chat Memory (RAM) with RAG Fusion"
run_count = 1

# Initialize Langsmith
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = "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"
os.environ["LANGSMITH_PROJECT"] = f"[{date}] VAA - {proj_name} {run_count}"

# Initialize LLM
REASONING = True
REASONING_CLASSIFIER = False

llm = ChatOllama(model="deepseek-r1:8b", validate_model_on_init=True, temperature=0.6, 
                 reasoning=True if REASONING else False)
simpler_llm = ChatOllama(model="deepseek-r1:8b", validate_model_on_init=True, 
                         reasoning=True if REASONING_CLASSIFIER else False)
emb = OllamaEmbeddings(model="bge-m3:567m")

num_queries = 3 # Number of additional queries to generate in RAG-Fusion

/var/folders/cf/1j9rc9v11rzf5wxcjw3w_wsm0000gp/T/ipykernel_13989/1199048702.py:35: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  emb = OllamaEmbeddings(model="bge-m3:567m")


In [2]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "academic_documents" if not SINGLE else "vaa_documents"

# Initialize retriever for queries. Get the workspace root directory
import pathlib

def get_project_root() -> pathlib.Path:
    current_file_dir = pathlib.Path(pathlib.Path.cwd()).resolve().parent
    if (current_file_dir / '.git').exists():
        return current_file_dir
    for parent in current_file_dir.parents:
        if (parent / '.git').exists():
            return parent
    return pathlib.Path.cwd() # Fallback to current working directory if .git not found

ROOT_DIR = get_project_root()

chroma_db_path = ROOT_DIR / "chroma_db"
print(f"Chroma DB path: {chroma_db_path}")

client = Client(Settings())
client = chromadb.PersistentClient(path=str(chroma_db_path))

vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=emb
)

client.get_collection(name=collection_name).count()

/var/folders/cf/1j9rc9v11rzf5wxcjw3w_wsm0000gp/T/ipykernel_13989/1086128921.py:24: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorStore = Chroma(


Chroma DB path: /Users/MarcussPC/Desktop/Temp/CAPSTONE/chroma_db


4320

### Query Transformation to Prevent Contextual Drift

**Problem**: When users ask follow-up questions like "What are the career paths for that programme?", the retriever doesn't know what "that programme" refers to, leading to retrieval of irrelevant documents.

**Solution**: Implement query transformation that converts follow-up questions into standalone, context-rich queries BEFORE retrieval:
- Uses chat history to understand what the user is referring to
- Reformulates ambiguous questions into clear, standalone queries
- Ensures the retriever gets complete context for accurate document retrieval

**Example**:
- Original follow-up: "What are the career paths for that programme?"
- Contextualized query: "What are the career paths for the BEng Scheme in IAIE programme?"

### RAG-Fusion Implementation

In [ ]:
# ================= Query Rewriting prompt =================
TRANSFORM_QUERY_PROMPT = \
"""
Given a chat history and the latest user question which might reference context in the chat history:

======
*Chat History*:
{chat_history}
======

Now, formulate a standalone question which can be understood without the chat history. 
*Do NOT answer the question*, just reformulate it if needed and otherwise return it as is.

*User Question*:
{question}

Reformulated standalone question:
"""
query_tra_prompt = PromptTemplate.from_template(TRANSFORM_QUERY_PROMPT)

# ================= RAG-Fusion prompt =================
RAG_FUSION_PROMPT = \
"""
You are a helpful assistant that generates multiple alternative queries based on a single input query.

Provide strictly {num_queries} alternative questions separated by newlines. Do not say anything else.

*User Question*: 
{question}

{num_queries} alternative questions:
"""
query_gen_prompt = PromptTemplate.from_template(RAG_FUSION_PROMPT)

# ================= DeepSeek-R1 prompt =================
LLM_PROMPT = \
"""
You are a professional academic advisor at The Hong Kong Polytechnic University. Given the following information:

======
*Previous Conversation*:
{chat_history}
======
*Context*:
{context}
======
*Student's Question*:
{question}
======

Please adhere to the following rules when answering the student's question:
1. Use the information from the previous conversation first, then the context, to answer the student's question.
2. Answer in the same language as the user query, e.g., English query, English answer.
3. Avoid saying "may", "maybe", or similar; be affirmative, confident, and decisive in your answers.
4. Avoid saying "based on the provided context", or similar; answer directly.
5. Say no if you cannot answer the question; *never fabricate a factually false answer*. Instead, ask for clarification.
6. Provide relevant URLs if necessary. However, *never fabricate non-existence URLs*. Only provide URLs from the context.
7. Provide advice to the student based on your answer and ask for any further enquiries, if applicable.

Now, give a helpful answer to the student!
"""
prompt = PromptTemplate.from_template(LLM_PROMPT)

In [5]:
# Initialize the memory saver
memory = MemorySaver()

# State class
class StateWithMemory(TypedDict):
    question: str
    contextualized_question: str    # For rewritten query
    queries: List[str]
    context: List[Document]
    answer: str
    chat_history: List[dict]        # Store chat history as list of message dicts
    thread_id: str                  # Unique ID for conversation thread

In [6]:
# Updated graph functions
def contextualize_question(state: StateWithMemory):
    question = state["question"]
    chat_history = state.get("chat_history", [])
    
    # If no chat history, the question is already standalone
    if not chat_history:
        print(f"[Query Transform] No history - using original: {question}")
        return {"contextualized_question": question}
    
    # Else, convert chat history to LangChain message format
    history = ""

    '''
    for msg in chat_history[-3:]:  # Use last 3 messages
        if msg["role"] == "user":
            history.append(HumanMessage(content=msg["content"]))
        elif msg["role"] == "assistant":
            history.append(AIMessage(content=msg["content"]))
    '''
    
    for msg in chat_history[-3:]:  # Use last 3 messages
        history += f"{msg['role'].capitalize()}: {msg['content']}\n\n"
    
    # Contextualize the question
    messages = query_tra_prompt.invoke({"chat_history": history, "question": question})
    response = simpler_llm.invoke(messages)
    
    contextualized_question = response.content.strip()
    
    print(f"[Query Transform] Original: {question}")
    print(f"[Query Transform] Contextualized: {contextualized_question}")
    
    return {"contextualized_question": contextualized_question}

def generate_queries(state: StateWithMemory):
    """Generate multiple queries using RAG-Fusion"""
    question = state.get("contextualized_question", state["question"])
    
    messages = query_gen_prompt.invoke({"question": question, "num_queries": num_queries})
    response = simpler_llm.invoke(messages)

    queries = response.content.strip().split("\n")
    queries = [q for q in queries if q.strip() != ""]
    
    print(f"[RAG-Fusion] Generated {len(queries)} queries from a question")
    
    return {"queries": queries}

def retrieve(state: StateWithMemory):
    """Retrieve documents using all generated queries"""
    all_docs = []
    
    print(f"[Retrieval] Retrieving for {len(state['queries'])} queries...")
    for idx, query in enumerate(state["queries"], 1):
        print(f"  Query {idx}: {query[:80]}...")
        retrieved_docs = vectorStore.similarity_search(query, k=4)
        all_docs.append(retrieved_docs)
    
    return {"context": all_docs}

def fuse_and_rerank(state: StateWithMemory, k: int = 60):
    """Fuse and rerank documents using Reciprocal Rank Fusion (RRF)"""
    fused_scores = {}
    
    for docs in state["context"]:
        for rank, doc in enumerate(docs):
            doc_str = doc.page_content
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            fused_scores[doc_str] += 1 / (rank + k)
    
    reranked_results = [
        (doc_str, score) for doc_str, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    
    unique_docs = {doc.page_content: doc for docs in state["context"] for doc in docs}.values()
    doc_map = {doc.page_content: doc for doc in unique_docs}
    
    num = 5  # Number of top documents to select
    reranked_docs = [doc_map[doc_str] for doc_str, _ in reranked_results[:num]]
    
    print(f"[Reranking] Selected top {len(reranked_docs)} documents after fusion")
    
    return {"context": reranked_docs}

def generate(state: StateWithMemory):
    """Generate answer considering chat history"""
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    
    # Format chat history for the prompt
    chat_history_str = ""
    if state.get("chat_history"):
        for msg in state["chat_history"][-3:]:  # Include last 3 messages
            chat_history_str += f"{msg['role'].capitalize()}: {msg['content']}\n\n"
    else:
        chat_history_str = "No previous conversation."
    
    messages = prompt.invoke({
        "question": state.get("contextualized_question", state["question"]),
        "context": docs_content,
        "chat_history": chat_history_str
    })
    
    # Gebnerate the answer, wooph
    response = llm.invoke(messages)
    
    # Update chat history after generating the answer
    new_history = state.get("chat_history", []).copy()
    new_history.append({"role": "user", "content": state.get("contextualized_question", state["question"])})
    new_history.append({"role": "assistant", "content": response.content})
    
    # Include reasoning if available
    if REASONING and response.additional_kwargs.get("reasoning_content"):
        answer = f"<think>\n{response.additional_kwargs.get('reasoning_content', '')}</think>\n\n{response.content}"
    else:
        answer = response.content
    
    return {
        "answer": answer,
        "chat_history": new_history
    }

# Build graph
def build_graph_with_memory():
    """With memory checkpointing and query transformation"""
    graph_builder = StateGraph(StateWithMemory)
    
    # Add query transformation as the first step
    graph_builder.add_node("contextualize_question", contextualize_question)
    graph_builder.add_node("generate_queries", generate_queries)
    graph_builder.add_node("retrieve", retrieve)
    graph_builder.add_node("fuse_and_rerank", fuse_and_rerank)
    graph_builder.add_node("generate", generate)
    
    # Updated edges with query transformation first
    graph_builder.add_edge(START, "contextualize_question")
    graph_builder.add_edge("contextualize_question", "generate_queries")
    graph_builder.add_edge("generate_queries", "retrieve")
    graph_builder.add_edge("retrieve", "fuse_and_rerank")
    graph_builder.add_edge("fuse_and_rerank", "generate")
    
    # Compile with memory checkpointer
    graph_with_memory = graph_builder.compile(checkpointer=memory)
    
    return graph_with_memory

# Create the graph
graph_with_memory = build_graph_with_memory()
print("\nGraph Flow: Contextualize → Generate Queries → Retrieve → Rerank (RRF) → Generate Answer")


Graph Flow: Contextualize → Generate Queries → Retrieve → Rerank (RRF) → Generate Answer


### Chat Memory Management Implementation

In [7]:
# Memory Management
def create_new_thread():
    """Create a new conversation thread"""
    return str(uuid.uuid4())

def get_chat_history(thread_id: str):
    """Retrieve chat history for a specific thread from memory"""
    try:
        # Get the state from the checkpoint
        config = {"configurable": {"thread_id": thread_id}}
        state = graph_with_memory.get_state(config)
        
        if state and state.values.get("chat_history"):
            return state.values["chat_history"]
        else:
            return []
    except Exception as e:
        print(f"Error retrieving chat history: {e}")
        return []

def display_chat_history(thread_id: str):
    """Display formatted chat history for a thread"""
    history = get_chat_history(thread_id)
    
    if not history:
        print(f"No chat history found for thread: {thread_id}")
        return
    
    print(f"\n{'='*60}")
    print(f"Chat History for Thread: {thread_id}")
    print(f"{'='*60}\n")
    
    for idx, msg in enumerate(history, 1):
        role = msg["role"].upper()
        content = msg["content"]
        print(f"{idx}. [{role}]")
        print(f"   {content[:200]}..." if len(content) > 200 else f"   {content}")
        print()

def clear_thread_memory(thread_id: str):
    """Clear all memory for a specific thread"""
    config = {"configurable": {"thread_id": thread_id}}
    
    try:
        # Update state with empty history
        graph_with_memory.update_state(
            config,
            {"chat_history": []}
        )
        print(f"Cleared memory for thread: {thread_id}")
        return True
    except Exception as e:
        print(f"Error clearing memory: {e}")
        return False

print("Memory management functions loaded successfully!")

Memory management functions loaded successfully!


### Example: Chat Memory with Multiple Conversations

In [8]:
# Example 1: Start a new conversation thread
thread_1 = create_new_thread()
print(f"Created Thread 1: {thread_1}\n")

# First query in the conversation
query_1 = "What is the programme of the Scheme of IAIE about? Why should I study this programme?"
config_1 = {"configurable": {"thread_id": thread_1}}

print(f"User: {query_1}")
result_1 = graph_with_memory.invoke(
    {
        "question": query_1,
        "chat_history": [],
        "thread_id": thread_1
    },
    config=config_1
)

print(f"\nAssistant: {result_1['answer'][:300]}...")
print(f"\n{'='*60}\n")

Created Thread 1: a6906b12-00de-46f5-bed5-707549d4d9b6

User: What is the programme of the Scheme of IAIE about? Why should I study this programme?
[Query Transform] No history - using original: What is the programme of the Scheme of IAIE about? Why should I study this programme?
[RAG-Fusion] Generated 3 queries from a question
[Retrieval] Retrieving for 3 queries...
  Query 1: What does the Scheme of IAIE programme cover, and what are its key features?  ...
  Query 2: Why is the IAIE programme a good choice for my studies?  ...
  Query 3: What are the benefits of enrolling in the IAIE Scheme programme?...
[Reranking] Selected top 5 documents after fusion

Assistant: <think>
Hmm, let's tackle this student's question about the IAIE programme at PolyU. First, I need to recall the provided context. The programme document lists electives like AAE4009, AAE4011, AMA3201, etc. These courses cover areas like data science, AI, computational methods, and more. The WIE com...




In [9]:
# Follow-up query in the same conversation
query_2 = "What are the career paths for that programme?"

print(f"User: {query_2}")

# Retrieve previous chat history from memory
previous_history = get_chat_history(thread_1)
print(f"\nRetrieved {len(previous_history)} previous messages from memory")

result_2 = graph_with_memory.invoke(
    {
        "question": query_2,
        "chat_history": previous_history,
        "thread_id": thread_1
    },
    config=config_1
)

print(f"\nAssistant: {result_2['answer'][:300]}...")
print(f"\nGenerated sub-queries:")
for q in result_2['queries']:
    print(f"  - {q}")
print(f"\n{'='*60}\n")

User: What are the career paths for that programme?

Retrieved 2 previous messages from memory
[Query Transform] Original: What are the career paths for that programme?
[Query Transform] Contextualized: What are the career paths available for graduates of the Bachelor of Engineering and Bachelor of Science Honours Scheme in Information and Artificial Intelligence Engineering?
[RAG-Fusion] Generated 3 queries from a question
[Retrieval] Retrieving for 3 queries...
  Query 1: What career opportunities are available for graduates of the Bachelor of Enginee...
  Query 2: How can graduates of the joint BE and BSc Honours program in Information and Art...
  Query 3: What are the professional trajectories for individuals who have completed the Ba...
[Reranking] Selected top 5 documents after fusion

Assistant: <think>
Okay, the user is asking about career paths for graduates of the IAIE programme. Let me start by recalling the previous conversation and the context provided.

In the previous r

In [10]:
# Display complete chat history for Thread 1
display_chat_history(thread_1)


Chat History for Thread: a6906b12-00de-46f5-bed5-707549d4d9b6

1. [USER]
   What is the programme of the Scheme of IAIE about? Why should I study this programme?

2. [ASSISTANT]
   The **Bachelor of Engineering and Bachelor of Science Honours Scheme in Information and Artificial Intelligence Engineering (IAIE)** at The Hong Kong Polytechnic University is designed to provide stud...

3. [USER]
   What are the career paths available for graduates of the Bachelor of Engineering and Bachelor of Science Honours Scheme in Information and Artificial Intelligence Engineering?

4. [ASSISTANT]
   Certainly! Here's the answer based on the provided context:

The **Bachelor of Engineering and Bachelor of Science Honours Scheme in Information and Artificial Intelligence Engineering (IAIE)** prepar...



### An Integrated Function (just for fun)

In [11]:
# Helper function for conversational RAG with memory
def chat_with_memory(question: str, thread_id: str = None):
    """
    Simplified function to chat with RAG system using memory
    
    Args:
        question: User's question
        thread_id: Optional thread ID. If None, creates a new thread
    
    Returns:
        dict: Result containing answer, thread_id, and chat history
    """
    # Create new thread if not provided
    if thread_id is None:
        thread_id = create_new_thread()
        print(f"Created new thread: {thread_id}")
    
    # Get existing chat history
    chat_history = get_chat_history(thread_id)
    
    # Configure thread
    config = {"configurable": {"thread_id": thread_id}}
    
    # Invoke the graph
    result = graph_with_memory.invoke(
        {
            "question": question,
            "chat_history": chat_history,
            "thread_id": thread_id
        },
        config=config
    )
    
    return {
        "answer": result["answer"],
        "thread_id": thread_id,
        "chat_history": result["chat_history"],
        "queries": result.get("queries", [])
    }

print("Helper function chat_with_memory() is ready to use!")

Helper function chat_with_memory() is ready to use!


### *Example: Complete Conversation Example with Memory

In [12]:
# Simulate a complete conversation with context awareness
print("Starting a new conversation...\n")

# Question 1
response_1 = chat_with_memory("What is the BEng Scheme in IAIE about? Any tips on achieving a good GPA in my study?")
my_thread = response_1["thread_id"]
print(f"[A1] {response_1['answer'][:200]}...\n")
print(f"{'='*60}\n")

# Question 2 - References previous context
response_2 = chat_with_memory("How can I become a professional engineer after studying that programme?", thread_id=my_thread)
print(f"[A2] {response_2['answer'][:200]}...\n")
print(f"{'='*60}\n")

# Question 3 - Further follow-up
response_3 = chat_with_memory("What are the scholarship options?", thread_id=my_thread)
print(f"[A3] {response_3['answer'][:200]}...\n")
print(f"{'='*60}\n")

# Display full conversation history
print("\nFull Conversation History:")
display_chat_history(my_thread)

Starting a new conversation...

Created new thread: af8440c1-99d1-4bb7-8be2-2b7a87a29ca2
[Query Transform] No history - using original: What is the BEng Scheme in IAIE about? Any tips on achieving a good GPA in my study?
[RAG-Fusion] Generated 3 queries from a question
[Retrieval] Retrieving for 3 queries...
  Query 1: What is the BEng Scheme in IAIE and what are effective strategies for maintainin...
  Query 2: Could you explain the structure of the BEng program at IAIE and provide advice o...
  Query 3: Please describe the BEng Scheme offered by IAIE and share recommendations for ex...
[Reranking] Selected top 5 documents after fusion
[A1] <think>
Okay, let's tackle this query. The user is asking about the BEng Scheme in IAIE and tips for achieving a good GPA. First, I need to check the provided context for relevant information.

Lookin...


[Query Transform] Original: How can I become a professional engineer after studying that programme?
[Query Transform] Contextualized: How can I 

### Demonstration: Query Transformation in Action

This example shows how query transformation prevents contextual drift:
- The first question establishes context about a specific programme
- Follow-up questions are automatically contextualized before retrieval
- This ensures the retriever gets accurate, standalone queries

In [13]:
# Demonstration: Query Transformation Preventing Contextual Drift
print("="*80)
print("DEMONSTRATION: Query Transformation in Action")
print("="*80)

# Start a new thread
demo_thread = create_new_thread()
config_demo = {"configurable": {"thread_id": demo_thread}}

# Question 1: Ask about a specific programme
print("\n[Step 1] Initial Question - Establishing Context")
print("-"*80)
query1 = "What is the BEng Scheme in Electrical Engineering about?"
print(f"User: {query1}\n")

result1 = graph_with_memory.invoke(
    {
        "question": query1,
        "chat_history": [],
        "thread_id": demo_thread
    },
    config=config_demo
)

print(f"\nAssistant: {result1['answer'][:250]}...\n")

# Question 2: Follow-up with ambiguous reference
print("\n[Step 2] Follow-up Question - Testing Query Transformation")
print("-"*80)
query2 = "What are the fields of career after graduating from that programme?"

print(f"User: {query2}")
print("\nℹ️  Note: This question is ambiguous - 'that programme' could refer to any programme.")
print("Query transformation will reformulate it into a standalone query.\n")

previous_history = get_chat_history(demo_thread)

result2 = graph_with_memory.invoke(
    {
        "question": query2,
        "chat_history": previous_history,
        "thread_id": demo_thread
    },
    config=config_demo
)

print(f"\n✅ Contextualized question used for retrieval: {result2.get('contextualized_question', 'N/A')}")
print(f"\nGenerated sub-queries from contextualized question:")
for idx, q in enumerate(result2['queries'], 1):
    print(f"  {idx}. {q}")

print(f"\nAssistant: {result2['answer'][:250]}...\n")

# Question 3: Another follow-up
print("\n[Step 3] Another Follow-up - Further Testing")
print("-"*80)
query3 = "What are the courses offered in that programme?"
print(f"User: {query3}")
print("\nℹ️  Note: Without query transformation, this would retrieve generic admission info.")
print("With transformation, it should specify which programme.\n")

previous_history = get_chat_history(demo_thread)

result3 = graph_with_memory.invoke(
    {
        "question": query3,
        "chat_history": previous_history,
        "thread_id": demo_thread
    },
    config=config_demo
)

print(f"\n✅ Contextualized question: {result3.get('contextualized_question', 'N/A')}")
print(f"\nAssistant: {result3['answer'][:250]}...\n")

print("="*80)
print("✅ Query Transformation Successfully Prevents Contextual Drift!")
print("="*80)

DEMONSTRATION: Query Transformation in Action

[Step 1] Initial Question - Establishing Context
--------------------------------------------------------------------------------
User: What is the BEng Scheme in Electrical Engineering about?

[Query Transform] No history - using original: What is the BEng Scheme in Electrical Engineering about?
[RAG-Fusion] Generated 3 queries from a question
[Retrieval] Retrieving for 3 queries...
  Query 1: What does the BEng program in Electrical Engineering entail?...
  Query 2: Could you explain the structure and curriculum of the BEng degree in Electrical ...
  Query 3: Please describe the key components and objectives of the BEng scheme in Electric...
[Reranking] Selected top 5 documents after fusion

Assistant: <think>
Hmm, the user is asking about the BEng Scheme in Electrical Engineering at The Hong Kong Polytechnic University. Let me recall the provided context.

Okay, the context includes details about the BEng (Hons) in Electrical Engineerin

### Before vs After: Query Transformation Impact

**Without Query Transformation (Contextual Drift):**
```
User: "What is the BEng in IAIE about?"
System retrieves: IAIE programme documents ✓

User: "What are the career paths for that programme?"
System retrieves: Random programme career documents ✗ (doesn't know which programme)
Result: Wrong programme information
```

**With Query Transformation (No Drift):**
```
User: "What is the BEng in IAIE about?"
System retrieves: IAIE programme documents ✓

User: "What are the career paths for that programme?"
→ Transformed to: "What are the career paths for BEng in IAIE?"
System retrieves: IAIE career documents ✓
Result: Correct programme-specific information
```

### System Architecture: Query Transformation Flow

```
┌─────────────────────────────────────────────────────────────────────┐
│                          USER INPUT                                 │
│              "What are the career paths for that programme?"        │
└─────────────────────────┬───────────────────────────────────────────┘
                          │
                          ▼
┌─────────────────────────────────────────────────────────────────────┐
│                 STEP 1: Query Transformation                        │
│  ┌───────────────────────────────────────────────────────────────┐ │
│  │ • Retrieve chat history from memory                           │ │
│  │ • Convert to LangChain message format                         │ │
│  │ • Use LLM to contextualize the question                       │ │
│  │                                                                │ │
│  │ Output: "What are the career paths for BEng in IAIE?"         │ │
│  └───────────────────────────────────────────────────────────────┘ │
└─────────────────────────┬───────────────────────────────────────────┘
                          │
                          ▼
┌─────────────────────────────────────────────────────────────────────┐
│                 STEP 2: RAG-Fusion Query Generation                 │
│  ┌───────────────────────────────────────────────────────────────┐ │
│  │ Generate 3 alternative queries from contextualized question:  │ │
│  │ 1. "What are the career paths for BEng in IAIE?"              │ │
│  │ 2. "What career opportunities exist for IAIE graduates?"      │ │
│  │ 3. "What jobs can I get with an IAIE degree?"                 │ │
│  └───────────────────────────────────────────────────────────────┘ │
└─────────────────────────┬───────────────────────────────────────────┘
                          │
                          ▼
┌─────────────────────────────────────────────────────────────────────┐
│                 STEP 3: Document Retrieval                          │
│  ┌───────────────────────────────────────────────────────────────┐ │
│  │ • Retrieve top-k documents for EACH query                     │ │
│  │ • All queries now contain "IAIE" context                      │ │
│  │ • Results: IAIE career documents ✓                            │ │
│  └───────────────────────────────────────────────────────────────┘ │
└─────────────────────────┬───────────────────────────────────────────┘
                          │
                          ▼
┌─────────────────────────────────────────────────────────────────────┐
│                 STEP 4: Reciprocal Rank Fusion                      │
│  ┌───────────────────────────────────────────────────────────────┐ │
│  │ • Fuse and rerank all retrieved documents                     │ │
│  │ • Select top 5 most relevant documents                        │ │
│  └───────────────────────────────────────────────────────────────┘ │
└─────────────────────────┬───────────────────────────────────────────┘
                          │
                          ▼
┌─────────────────────────────────────────────────────────────────────┐
│                 STEP 5: Answer Generation                           │
│  ┌───────────────────────────────────────────────────────────────┐ │
│  │ • Generate answer using reranked documents + chat history     │ │
│  │ • Update chat history with new Q&A pair                       │ │
│  │ • Persist to memory checkpoint                                │ │
│  └───────────────────────────────────────────────────────────────┘ │
└─────────────────────────┬───────────────────────────────────────────┘
                          │
                          ▼
┌─────────────────────────────────────────────────────────────────────┐
│                    ACCURATE ANSWER                                  │
│   "IAIE graduates can pursue careers in operations management,      │
│    supply chain optimization, industrial automation..."             │
└─────────────────────────────────────────────────────────────────────┘
```

**Key Insight**: Query transformation at the START ensures all downstream components work with accurate, context-rich queries.

### Summary of Features Implemented

This implementation provides:

1. **Query Transformation (NEW)**: Prevents contextual drift in follow-up questions
   - Automatically converts ambiguous follow-up questions into standalone, context-rich queries
   - Uses chat history to understand references like "that programme", "it", "those", etc.
   - Ensures retriever gets accurate context for document retrieval
   - Flow: User Question → Contextualize → RAG-Fusion → Retrieve → Rerank → Answer

2. **Persistent Chat History**: Using LangChain's `MemorySaver` to store conversation history in memory
   - Each conversation thread has a unique ID
   - History persists across multiple queries within the same session
   
3. **Multi-Thread Support**: Manage multiple independent conversations
   - Each thread maintains separate context
   - No cross-contamination between conversations
   
4. **RAG-Fusion Integration**: Memory-aware query generation
   - Recent chat history considered when generating alternative queries
   - Context-aware document retrieval and ranking

**Key Functions:**
- `create_new_thread()`: Start a new conversation
- `chat_with_memory()`: Simplified interface for conversational RAG
- `get_chat_history()`: Retrieve conversation history
- `display_chat_history()`: Show formatted history
- `clear_thread_memory()`: Reset a conversation

**How Query Transformation Solves Contextual Drift:**
- **Before**: "What are the career paths for that programme?" → Retrieves random programme documents
- **After**: "What are the career paths for that programme?" → Transformed to "What are the career paths for the BEng Scheme in IAIE programme?" → Retrieves correct documents

**Note**: `InMemorySaver` stores data in RAM only. For production use with persistent storage across restarts, consider using `SqliteSaver` or `PostgresSaver`.